# Pruebas — VentasApp (notebook experimental)

## Contexto del proyecto

**Que datos tiene:** un CSV de ventas (data/raw/Base de datos de Ventas.csv)
con transacciones de 5 productos electronicos (Laptop, Smartphone, Tablet,
Monitor, Teclado) entre 2019 y 2023: año, mes, cliente, producto, unidades
vendidas, precio unitario e ingresos. ~5,000 filas.

**Que le hace el ETL:**
- *Transformacion* (`modulos/etl_ventas.py`): separa el Cliente_ID del texto
  original, traduce productos y meses al español, renombra columnas.
- *Validacion* (`modulos/schema.py` + `modulos/validacion.py`): Pandera
  revisa reglas de negocio (Ingresos = Unidades x Precio, valores positivos,
  mes valido, etc.) y descarta filas que no cumplen, sin frenar el pipeline.
- *Feature engineering* (`modulos/feature_engineering.py`): agrega historial
  por cliente, recencia, participacion/ranking de producto y segmentacion de
  clientes por valor.
- *Load* (`etl_ventas_main.py`): guarda el resultado en output/ventas.parquet
  y output/ventas.duckdb.

**Para que sirve / que se analiza:** el dashboard (`dashboard.py`) usa este
dataset para responder preguntas de negocio — que producto genera mas
ingresos, como evoluciona el revenue mes a mes, si precio y volumen se
relacionan, quienes son los clientes mas valiosos, y que tan correlacionadas
estan unidades/precio/ingresos entre si.

Este notebook (`Pruebas.ipynb`) importa directamente de `modulos/` en vez de
reescribir la logica: lo que se prueba aqui es el mismo codigo que corre
`etl_ventas_main.py`, sin que se desincronicen dos copias.

Contenido de este notebook (5 preguntas de negocio: 2 gráficos + 3 tablas):
1. Carga del dataset ya procesado (`output/ventas.parquet`)
2. [Gráfico] ¿Quiénes son mis clientes más valiosos?
3. [Gráfico] ¿En qué meses/años se concentran los ingresos?
4. [Tabla] ¿Qué clientes están en riesgo de abandono?
5. [Tabla] ¿Cómo se compara el desempeño de cada producto?
6. [Tabla] ¿Qué segmento de clientes concentra más ingresos?
7. Validación de esquema con Pandera (probando el schema real contra datos crudos)
8. Optimización de memoria y tiempos de ejecución

## 0. Setup — importar los módulos de producción

In [ ]:
import sys
from pathlib import Path

# Asegura que la raíz del proyecto (donde está este notebook) esté en el path,
# para poder hacer `import modulos...` sin importar desde dónde se lance Jupyter.
ROOT = Path.cwd()
if not (ROOT / "modulos").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from config.config import PARQUET_PATH, RAW_DIR
from modulos.schema import schema_ventas
from modulos.validacion import validate_dataframe

print("Imports listos. ROOT:", ROOT)

## 1. Cargar el dataset ya procesado

In [ ]:
df = pd.read_parquet(PARQUET_PATH)
print(df.shape)
df.head()

## 2. Gráfico — ¿Quiénes son mis clientes más valiosos? (recap)

Ranking horizontal de los 10 clientes con mayor `monto_total` histórico
(columna que ya viene calculada por `feature_engineering.py`).

In [ ]:
top10 = (df.drop_duplicates(subset="Cliente_ID")
           .sort_values("monto_total", ascending=False)
           .head(10)
           .sort_values("monto_total"))

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

bars = ax.barh(top10["Nombre_Cliente"], top10["monto_total"], color="#2a78d6", height=0.6, zorder=3)

for bar, valor in zip(bars, top10["monto_total"]):
    ax.text(bar.get_width() + top10["monto_total"].max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f"{valor:,.0f}", va="center", ha="left", color="#0b0b0b", fontsize=9)

ax.set_title("Top 10 clientes por monto total de compra", color="#0b0b0b", fontsize=13, loc="left", pad=12)
ax.set_xlabel("Monto total (Ingresos)", color="#52514e")
ax.tick_params(colors="#52514e")
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#c3c2b7")
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

## 3. Gráfico nuevo — ¿En qué meses y años se concentran los ingresos?

Pregunta de negocio: si tuviera que reforzar inventario o campañas, ¿en qué
combinación de año + mes históricamente entran más ingresos? Un heatmap
Año × Mes responde eso de un vistazo, algo que un bar chart o una serie de
tiempo simple no muestran igual de claro (no se ve si el patrón mensual se
repite entre años o si un año concreto explica el pico).

In [ ]:
ORDEN_MESES = [
    "Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre",
]

pivot = (
    df.groupby(["Año", "Mes"])["Ingresos"]
    .sum()
    .reset_index()
    .pivot(index="Año", columns="Mes", values="Ingresos")
    .reindex(columns=ORDEN_MESES)
)

fig, ax = plt.subplots(figsize=(11, 3.5))
fig.patch.set_facecolor("#fcfcfb")

im = ax.imshow(pivot.values, cmap="Blues", aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right", color="#52514e")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, color="#52514e")

# Etiquetas de valor dentro de cada celda
vmax = pivot.values.max()
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        valor = pivot.values[i, j]
        color_texto = "#ffffff" if valor > vmax * 0.6 else "#0b0b0b"
        ax.text(j, i, f"{valor/1000:,.0f}k", ha="center", va="center", color=color_texto, fontsize=8)

ax.set_title("Ingresos por Año x Mes", color="#0b0b0b", fontsize=13, loc="left", pad=12)
cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.ax.tick_params(colors="#52514e")

plt.tight_layout()
plt.show()

## 4. Pregunta de negocio — ¿Qué clientes están en riesgo de abandono?

Los clientes que llevan más meses sin comprar (`meses_desde_ultima_compra`)
son candidatos a una campaña de reactivación — sobre todo si además son de
`segmento_valor` alto, porque ahí se pierde más ingreso si no vuelven.

In [ ]:
riesgo_abandono = (
    df.drop_duplicates(subset="Cliente_ID")
    [["Cliente_ID", "Nombre_Cliente", "segmento_valor", "monto_total", "meses_desde_ultima_compra"]]
    .sort_values("meses_desde_ultima_compra", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
riesgo_abandono

## 5. Pregunta de negocio — ¿Cómo se compara el desempeño de cada producto?

Un scorecard de una fila por producto: unidades, ingresos, precio promedio,
ticket promedio, clientes únicos alcanzados, participación en el revenue
total y ranking. Útil para decidir en qué producto enfocar inventario o
promociones.

In [ ]:
scorecard_producto = (
    df.groupby("Producto")
    .agg(
        unidades_totales=("Unidades_Vendidas", "sum"),
        ingresos_totales=("Ingresos", "sum"),
        precio_promedio=("Precio_Unitario", "mean"),
        ticket_promedio=("Ingresos", "mean"),
        clientes_unicos=("Cliente_ID", "nunique"),
        participacion_pct=("participacion_pct", "first"),
        ranking_producto=("ranking_producto", "first"),
    )
    .round(2)
    .sort_values("ranking_producto")
)
scorecard_producto

## 6. Pregunta de negocio — ¿Qué segmento de clientes concentra más ingresos?

Cada segmento de valor (`segmento_valor`, cuartiles de `monto_total`) tiene
por definición el mismo número de clientes (25% cada uno). La pregunta real
es si también se reparten el ingreso por igual, o si un segmento pequeño
concentra desproporcionadamente el revenue (patrón tipo Pareto/80-20).

In [ ]:
clientes_unicos = df.drop_duplicates(subset="Cliente_ID")
ingreso_total = clientes_unicos["monto_total"].sum()
total_clientes = len(clientes_unicos)

concentracion_valor = (
    clientes_unicos.groupby("segmento_valor")
    .agg(num_clientes=("Cliente_ID", "count"), ingreso_segmento=("monto_total", "sum"))
)
concentracion_valor["pct_clientes"] = (concentracion_valor["num_clientes"] / total_clientes * 100).round(1)
concentracion_valor["pct_ingreso"] = (concentracion_valor["ingreso_segmento"] / ingreso_total * 100).round(1)
concentracion_valor = concentracion_valor.reindex(["Alto", "Medio-alto", "Medio-bajo", "Bajo"])

concentracion_valor

## 7. Validar esquema con Pandera (usando el schema real de producción)

Esto no revalida `output/ventas.parquet` (ya pasó por `validate_dataframe` en
el pipeline) — lo interesante es probar `schema_ventas` contra el **CSV
crudo recién extraído/transformado**, como si fuera una corrida nueva, para
ver el reporte de rechazos en vivo.

In [ ]:
from modulos.etl_ventas import extraer_y_limpiar

CSV_PATH = RAW_DIR / "Base de datos de Ventas.csv"

df_polars = extraer_y_limpiar(CSV_PATH)
df_validado, resumen_rechazos = validate_dataframe(df_polars)

print(f"Filas de entrada: {df_polars.height}")
print(f"Filas válidas   : {df_validado.height}")
print(f"Resumen de rechazos: {resumen_rechazos or 'sin rechazos'}")

## 8. Optimización de memoria y tiempos de ejecución

Dos preguntas separadas:
- **Memoria**: ¿cuánto pesa el DataFrame con los tipos "por defecto" vs. con
  tipos ajustados al rango real de cada columna?
- **Tiempo**: ¿lectura eager vs. lazy en Polars, y Polars vs. pandas para la
  misma agregación?

Con ~5,000 filas las diferencias son pequeñas (dataset chico), así que
también se repite la prueba de tiempos sobre una versión inflada 200x del
dataset, para que el efecto sea visible — el punto es el método, no el
tamaño actual de `VentasApp`.

In [ ]:
# --- Memoria: tipos por defecto vs. tipos optimizados -----------------------

df_pl = pl.read_parquet(PARQUET_PATH)

mem_original_mb = df_pl.estimated_size("mb")
print(f"Memoria con tipos originales: {mem_original_mb:.3f} MB")
print(df_pl.schema)

In [ ]:
# Downcasting: enteros a rangos mas chicos, floats a 32 bits,
# strings de baja cardinalidad (Producto, Mes) a categorico.

df_pl_opt = df_pl.with_columns([
    pl.col("Año").cast(pl.Int16),
    pl.col("Unidades_Vendidas").cast(pl.Int16),
    pl.col("Cliente_ID").cast(pl.Int32),
    pl.col("Precio_Unitario").cast(pl.Float32),
    pl.col("Ingresos").cast(pl.Float32),
    pl.col("Producto").cast(pl.Categorical),
    pl.col("Mes").cast(pl.Categorical),
])

mem_optimizada_mb = df_pl_opt.estimated_size("mb")
reduccion = 1 - (mem_optimizada_mb / mem_original_mb)

print(f"Memoria con tipos optimizados: {mem_optimizada_mb:.3f} MB")
print(f"Reduccion: {reduccion:.1%}")

In [ ]:
# --- Tiempo: lectura eager vs. lazy (Polars) --------------------------------

import timeit

t_eager = timeit.timeit(lambda: pl.read_csv(CSV_PATH), number=5) / 5
t_lazy = timeit.timeit(lambda: pl.scan_csv(CSV_PATH).collect(), number=5) / 5

print(f"Lectura eager (pl.read_csv)         : {t_eager*1000:.2f} ms (promedio de 5 corridas)")
print(f"Lectura lazy  (pl.scan_csv + collect): {t_lazy*1000:.2f} ms (promedio de 5 corridas)")
print(
    "-> En un CSV chico como este la diferencia es minima. "
    "La ventaja de scan_csv/lazy aparece con archivos grandes o cuando se "
    "encadenan filtros/selects antes de leer todo a memoria (predicate/projection pushdown)."
)

In [ ]:
# --- Tiempo: Polars vs. pandas para la misma agregacion ---------------------

df_pd = df_pl.to_pandas()

t_polars = timeit.timeit(
    lambda: df_pl.group_by("Producto").agg(pl.col("Ingresos").mean()),
    number=20,
) / 20

t_pandas = timeit.timeit(
    lambda: df_pd.groupby("Producto")["Ingresos"].mean(),
    number=20,
) / 20

print(f"Polars group_by().agg(): {t_polars*1000:.3f} ms (promedio de 20 corridas)")
print(f"Pandas groupby()       : {t_pandas*1000:.3f} ms (promedio de 20 corridas)")

if t_polars < t_pandas:
    print(f"-> Polars fue {t_pandas/t_polars:.1f}x mas rapido")
else:
    print(f"-> Pandas fue {t_polars/t_pandas:.1f}x mas rapido (normal en datasets chicos, Polars tiene mas overhead fijo)")

**Repitiendo la comparación de tiempos sobre un dataset 200x más grande**
(mismo contenido, concatenado), para ver el efecto a una escala más realista:

In [ ]:
N = 200

df_pl_grande = pl.concat([df_pl] * N)
df_pd_grande = df_pl_grande.to_pandas()

print(f"Filas del dataset inflado: {df_pl_grande.height:,}")

t_polars_grande = timeit.timeit(
    lambda: df_pl_grande.group_by("Producto").agg(pl.col("Ingresos").mean()),
    number=10,
) / 10

t_pandas_grande = timeit.timeit(
    lambda: df_pd_grande.groupby("Producto")["Ingresos"].mean(),
    number=10,
) / 10

print(f"Polars group_by().agg(): {t_polars_grande*1000:.2f} ms (promedio de 10 corridas)")
print(f"Pandas groupby()       : {t_pandas_grande*1000:.2f} ms (promedio de 10 corridas)")

if t_polars_grande < t_pandas_grande:
    print(f"-> Con {df_pl_grande.height:,} filas, Polars fue {t_pandas_grande/t_polars_grande:.1f}x mas rapido")
else:
    print(f"-> Con {df_pl_grande.height:,} filas, Pandas fue {t_polars_grande/t_pandas_grande:.1f}x mas rapido")

del df_pl_grande, df_pd_grande  # liberar memoria, ya cumplió su proposito de prueba